# Attention Mechanisms aur Transformers

Recurrent Networks ki ek badi kamzori yeh hai ki sequence ke saare words ka result par same impact hota hai. Is wajah se sequence-to-sequence tasks, jaise Named Entity Recognition aur Machine Translation, ke liye standard LSTM encoder-decoder models ki performance kam ho jaati hai. Asal mein, input sequence ke kuch khaas words ka sequential output par baaki words se zyada impact hota hai.

Maan lijiye humare paas ek sequence-to-sequence model hai, jaise machine translation. Ise do recurrent networks se implement kiya jaata hai, jahan ek network (**encoder**) input sequence ko ek hidden state mein compress karta hai, aur doosra network (**decoder**) is hidden state ko translated result mein expand karta hai. Is approach ki problem yeh hai ki network ki final state ko sentence ki shuruaat yaad rakhne mein dikkat hoti hai, jisse lambe sentences par model ki quality kharab ho jaati hai.

**Attention Mechanisms** RNN ke har output prediction par har input vector ke contextual impact ko weigh karne ka ek tareeka dete hain. Ise implement karne ka tareeka yeh hai ki input RNN ki intermediate states aur output RNN ke beech shortcuts banaye jaate hain. Is tarah, jab output symbol $y_t$ generate hota hai, toh hum saari input hidden states $h_i$ ko alag-alag weight coefficients $\alpha_{t,i}$ ke saath dhyan mein rakhenge.

![Encoder/decoder model ka diagram jisme additive attention layer hai](../../../../../lessons/5-NLP/18-Transformers/images/encoder-decoder-attention.png)
*Additive attention mechanism ke saath encoder-decoder model, [Bahdanau et al., 2015](https://arxiv.org/pdf/1409.0473.pdf) se, [is blog post](https://lilianweng.github.io/lil-log/2018/06/24/attention-attention.html) se liya gaya*

Attention matrix $\{\alpha_{i,j}\}$ dikhata hai ki output sequence mein kisi diye gaye word ko generate karne mein kaunse input words kitna role nibhate hain. Neeche is tarah ke matrix ka ek example diya gaya hai:

![RNNsearch-50 dwara paya gaya ek sample alignment dikhane wali image, Bahdanau - arviz.org se li gayi](../../../../../lessons/5-NLP/18-Transformers/images/bahdanau-fig3.png)

*[Bahdanau et al., 2015](https://arxiv.org/pdf/1409.0473.pdf) (Fig.3) se li gayi image]*

Attention mechanisms aaj ke ya lagbhag aaj ke Natural Language Processing mein state-of-the-art techniques ke liye zimmedar hain. Lekin, attention add karne se model ke parameters ki sankhya kaafi badh jaati hai, jisse RNNs ke saath scaling problems aati hain. RNNs ko scale karne mein ek badi rukawat yeh hai ki model ki recurrent nature ki wajah se training ko batch aur parallelize karna mushkil ho jaata hai. RNN mein sequence ke har element ko ek ke baad ek process karna padta hai, isliye ise aasaani se parallelize nahi kiya ja sakta.

Attention mechanisms ko apnane aur is rukawat ne aaj ke state-of-the-art Transformer models ke banne ka raasta khola, jinhe hum BERT se OpenGPT3 tak jaante aur use karte hain.

## Transformer Models

Har prediction ke context ko agle evaluation step tak forward karne ki jagah, **Transformer models** **positional encodings** aur attention ka use karke diye gaye input ka context ek specified text window ke andar capture karte hain. Neeche di gayi image dikhati hai ki positional encodings aur attention use karke window ke andar context kaise capture kiya ja sakta hai.

![Animated GIF jo dikhata hai ki Transformer models mein evaluation kaise hota hai.](../../../../../lessons/5-NLP/18-Transformers/images/transformer-animated-explanation.gif)

Kyunki har input position ko independently har output position par map kiya jaata hai, Transformers RNNs ke muqable behtar parallelism dete hain, jisse bahut bade aur zyada expressive language models possible hote hain. Har attention head ka use words ke beech alag-alag relationships seekhne ke liye kiya ja sakta hai, jisse downstream Natural Language Processing tasks behtar hote hain.

**BERT** (Bidirectional Encoder Representations from Transformers) ek bahut bada multi-layer Transformer network hai, jisme *BERT-base* ke liye 12 layers aur *BERT-large* ke liye 24 layers hoti hain. Is model ko pehle bade text data (Wikipedia + books) par unsupervised training (sentence mein chhupe hue words ko predict karna) se train kiya jaata hai. Pre-training ke dauran, model language ki important samajh haasil kar leta hai, jise phir doosre datasets ke saath fine-tuning ke through use kiya ja sakta hai. Is process ko **Transfer Learning** kehte hain.

![Image http://jalammar.github.io/illustrated-bert/ se](../../../../../lessons/5-NLP/18-Transformers/images/jalammarBERT-language-modeling-masked-lm.png)

Transformer architecture ke kai variants hain, jaise BERT, DistilBERT, BigBird, OpenGPT3 aur doosre, jinhe fine-tune kiya ja sakta hai. [HuggingFace package](https://github.com/huggingface/) PyTorch ke saath in mein se kai architectures ko train karne ke liye ek repository deta hai.

## Text Classification ke liye BERT ka use

Chaliye dekhte hain ki hum apne traditional task ko solve karne ke liye pre-trained BERT model ka use kaise kar sakte hain: sequence classification. Hum apne original AG News dataset ko classify karenge.

Sabse pehle, HuggingFace library aur hamara dataset load karte hain:

In [ ]:
import torch
import torchtext
from torchnlp import *
import transformers
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_len = len(vocab)

Kyunki hum pre-trained BERT model use karenge, humein ek special tokenizer use karna hoga. Sabse pehle, hum pre-trained BERT model se juda hua tokenizer load karenge.

HuggingFace library mein pre-trained models ki ek repository hai, jise aap sirf unke naam `from_pretrained` function ke arguments mein dekar use kar sakte hain. Model ke liye zaroori saari binary files automatically download ho jaayengi.

Lekin, kuch situations mein aapko apne khud ke models load karne pad sakte hain. Aise cases mein, aap woh directory specify kar sakte hain jisme saari related files hon, jaise tokenizer ke parameters, model parameters ke saath `config.json` file, binary weights, aadi.

In [ ]:
# To load the model from Internet repository using model name. 
# Use this if you are running from your own copy of the notebooks
bert_model = 'bert-base-uncased' 

# To load the model from the directory on disk. Use this for Microsoft Learn module, because we have
# prepared all required files for you.
bert_model = './bert'

tokenizer = transformers.BertTokenizer.from_pretrained(bert_model)

MAX_SEQ_LEN = 128
PAD_INDEX = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)
UNK_INDEX = tokenizer.convert_tokens_to_ids(tokenizer.unk_token)

`tokenizer` object mein `encode` function hota hai jise seedhe text ko encode karne ke liye use kiya ja sakta hai:

In [ ]:
tokenizer.encode('PyTorch is a great framework for NLP')

Phir, chaliye aise iterators banate hain jinhe hum training ke dauran data access karne ke liye use karenge. Kyunki BERT apna khud ka encoding function use karta hai, humein ek padding function define karna hoga jo pehle define kiye gaye `padify` jaisa ho:

In [ ]:
def pad_bert(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [tokenizer.encode(x[1]) for x in b]
    # compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0] for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, collate_fn=pad_bert, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, collate_fn=pad_bert)

Hamare case mein, hum pre-trained BERT model use karenge jise `bert-base-uncased` kehte hain. Chaliye model ko `BertForSequenceClassfication` package use karke load karte hain. Yeh ensure karta hai ki hamare model mein classification ke liye zaroori structure pehle se hai, jisme final classifier bhi shamil hai. Aapko ek warning message dikhega jisme likha hoga ki final classifier ke weights initialize nahi kiye gaye hain, aur model ko pre-training ki zaroorat hogi - yeh bilkul theek hai, kyunki yahi toh hum karne ja rahe hain!

In [ ]:
model = transformers.BertForSequenceClassification.from_pretrained(bert_model,num_labels=4).to(device)

Ab hum training shuru karne ke liye ready hain! Kyunki BERT pehle se pre-trained hai, hum bahut chhote learning rate se shuru karna chahte hain taaki initial weights kharab na ho jaayein.

Saara mushkil kaam `BertForSequenceClassification` model karta hai. Jab hum training data par model ko call karte hain, toh yeh input minibatch ke liye loss aur network output dono return karta hai. Hum parameter optimization ke liye loss use karte hain (`loss.backward()` backward pass karta hai), aur `out` ka use training accuracy calculate karne ke liye karte hain, jo obtained labels `labs` (jo `argmax` use karke calculate hote hain) ko expected `labels` se compare karta hai.

Process ko control karne ke liye, hum kai iterations mein loss aur accuracy ko accumulate karte hain, aur har `report_freq` training cycles ke baad unhe print karte hain.

Yeh training shayad kaafi time legi, isliye hum iterations ki sankhya ko limit karte hain.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

report_freq = 50
iterations = 500 # make this larger to train for longer time!

model.train()

i,c = 0,0
acc_loss = 0
acc_acc = 0

for labels,texts in train_loader:
    labels = labels.to(device)-1 # get labels in the range 0-3         
    texts = texts.to(device)
    loss, out = model(texts, labels=labels)[:2]
    labs = out.argmax(dim=1)
    acc = torch.mean((labs==labels).type(torch.float32))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    acc_loss += loss
    acc_acc += acc
    i+=1
    c+=1
    if i%report_freq==0:
        print(f"Loss = {acc_loss.item()/c}, Accuracy = {acc_acc.item()/c}")
        c = 0
        acc_loss = 0
        acc_acc = 0
    iterations-=1
    if not iterations:
        break

Aap dekh sakte hain (khaaskar agar aap iterations ki sankhya badhate hain aur thoda zyada wait karte hain) ki BERT classification humein kaafi acchi accuracy deta hai! Iska kaaran yeh hai ki BERT pehle se hi language ki structure ko kaafi acchi tarah samajhta hai, aur humein sirf final classifier ko fine-tune karna hota hai. Lekin, kyunki BERT ek bada model hai, poora training process kaafi time leta hai aur iske liye serious computational power chahiye hoti hai! (GPU, aur behtar hoga ki ek se zyada GPU hon).

> **Note:** Hamare example mein, humne sabse chhote pre-trained BERT models mein se ek ka use kiya hai. Bade models available hain, jo shayad behtar results de sakte hain.

## Model Performance ka Evaluation

Ab hum apne model ki performance ko test dataset par evaluate kar sakte hain. Evaluation loop training loop jaisa hi hai, lekin humein yeh nahi bhoolna chahiye ki model ko evaluation mode mein switch karne ke liye `model.eval()` call karna zaroori hai.

In [ ]:
model.eval()
iterations = 100
acc = 0
i = 0
for labels,texts in test_loader:
    labels = labels.to(device)-1      
    texts = texts.to(device)
    _, out = model(texts, labels=labels)[:2]
    labs = out.argmax(dim=1)
    acc += torch.mean((labs==labels).type(torch.float32))
    i+=1
    if i>iterations: break
        
print(f"Final accuracy: {acc.item()/i}")

## Key Takeaways

Is unit mein, humne dekha ki **transformers** library se pre-trained language model lena aur use hamare text classification task ke liye adapt karna kitna aasaan hai. Isi tarah, BERT models ka use entity extraction, question answering aur doosre NLP tasks ke liye bhi kiya ja sakta hai.

Transformer models NLP mein abhi ki sabse advanced technology ko represent karte hain, aur zyaadatar cases mein, jab aap custom NLP solutions implement karna shuru karte hain, toh yeh pehla option hona chahiye jiske saath aap experiment karein. Lekin, agar aap advanced neural models banana chahte hain, toh is module mein discuss kiye gaye recurrent neural networks ke fundamental principles ko samajhna bahut important hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->